In [25]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils import data
torch.manual_seed(42)

epoch = 300
batch_size = 64
num_steps = 20
lr = 0.1
num_examples = 600

data_path = "../data/fra-eng/fra.txt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)

device: cuda


In [26]:
def data_read(path: str) -> str:
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def preprocess_nmt(text: str) -> str:
    text = (
        text
        .replace("\u202f", " ")
        .replace("\xa0", " ")
        .lower()
    )

    result = []

    for i, char in enumerate(text):
        if (
            char in ",.!?"
            and i > 0
            and text[i - 1] != " "
        ):
            result.append(" ")

        result.append(char)

    return "".join(result)


def tokenize_nmt(text: str, max_text: int = 600):
    source = []
    target = []

    for line in text.splitlines():
        parts = line.split("\t")

        if len(parts) < 2:
            continue

        source.append(parts[0].split())
        target.append(parts[1].split())

        if max_text is not None and len(source) >= max_text:
            break

    return source, target

In [27]:
def vocab_maker(tokens, min_freq: int = 2):
    counter = Counter()

    for line in tokens:
        counter.update(line)

    token_freq = sorted(
        counter.items(),
        key=lambda item: (-item[1], item[0])
    )

    idx_to_token = [
        "<unk>",
        "<pad>",
        "<bos>",
        "<eos>",
    ]

    for token, freq in token_freq:
        if freq < min_freq:
            continue

        if token not in idx_to_token:
            idx_to_token.append(token)

    token_to_idx = {
        token: idx
        for idx, token in enumerate(idx_to_token)
    }

    return {
        "idx_to_token": idx_to_token,
        "token_to_idx": token_to_idx,
    }


def truncate_pad(line, step_num: int, pad_idx: int):
    if len(line) > step_num:
        return line[:step_num]

    return line + [pad_idx] * (step_num - len(line))


def build_array_nmt(lines, vocab, step_num: int):
    token_to_idx = vocab["token_to_idx"]

    unk_idx = token_to_idx["<unk>"]
    pad_idx = token_to_idx["<pad>"]
    eos_idx = token_to_idx["<eos>"]

    array = []

    for line in lines:
        line_idx = [
            token_to_idx.get(token, unk_idx)
            for token in line
        ]

        line_idx.append(eos_idx)
        line_idx = truncate_pad(
            line_idx,
            step_num,
            pad_idx,
        )

        array.append(line_idx)

    array = torch.tensor(array, dtype=torch.long)
    valid_len = (array != pad_idx).sum(dim=1)

    return array, valid_len


def load_data_nmt(
    batch_size: int,
    step_num: int,
    max_examples: int = 600,
):
    raw_text = data_read(data_path)
    text = preprocess_nmt(raw_text)
    source, target = tokenize_nmt(text, max_examples)

    src_vocab = vocab_maker(source, min_freq=2)
    tgt_vocab = vocab_maker(target, min_freq=2)

    src_array, src_valid_len = build_array_nmt(
        source,
        src_vocab,
        step_num,
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target,
        tgt_vocab,
        step_num,
    )

    dataset = data.TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len,
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    return train_iter, src_vocab, tgt_vocab

In [28]:
train_iter, src_vocab, tgt_vocab = load_data_nmt(
    batch_size,
    num_steps,
    num_examples,
)

src_pad_idx = src_vocab["token_to_idx"]["<pad>"]
tgt_pad_idx = tgt_vocab["token_to_idx"]["<pad>"]
tgt_bos_idx = tgt_vocab["token_to_idx"]["<bos>"]
tgt_eos_idx = tgt_vocab["token_to_idx"]["<eos>"]

X, X_valid_len, Y, Y_valid_len = next(iter(train_iter))

print("X shape:", X.shape)
print("X_valid_len shape:", X_valid_len.shape)
print("Y shape:", Y.shape)
print("Y_valid_len shape:", Y_valid_len.shape)
print("src vocab size:", len(src_vocab["idx_to_token"]))
print("tgt vocab size:", len(tgt_vocab["idx_to_token"]))

X shape: torch.Size([64, 20])
X_valid_len shape: torch.Size([64])
Y shape: torch.Size([64, 20])
Y_valid_len shape: torch.Size([64])
src vocab size: 184
tgt vocab size: 201


In [29]:
def mask_softmax(scores, valid_len):
    # scores: (batch*head_num, query_num, key_num)
    # valid_len: (batch*head_num,)
    positions = torch.arange(scores.shape[-1], device=scores.device)
    mask = positions.reshape(1, 1, -1) >= valid_len.reshape(-1, 1, 1)
    scores = scores.masked_fill(mask, float("-inf"))
    return torch.softmax(scores, dim=-1)
class attention_module(nn.Module):
    def __init__(self,query_size,key_size,value_size,head_num,attention_hidden):
        super().__init__()
        self.w_q=nn.Linear(query_size,attention_hidden)
        self.w_k=nn.Linear(key_size,attention_hidden)
        self.w_v=nn.Linear(value_size,attention_hidden)
        self.head_num=head_num
        self.head_size=attention_hidden//head_num
        self.w_o=nn.Linear(attention_hidden,attention_hidden)
    def split(self,x):
        batch=x.shape[0]
        step=x.shape[1]
        x=x.reshape(batch,step,self.head_num,self.head_size)
        x=x.permute(0,2,1,3)
        x=x.reshape(-1,step,self.head_size)
        return x
    def merge(self,x):
        batch=x.shape[0]//self.head_num
        step=x.shape[1]
        x=x.reshape(batch,self.head_num,step,self.head_size)
        x=x.permute(0,2,1,3)
        x=x.reshape(batch,step,-1)
        return x
    def forward(self,query,key,value,valid_len,flag=False):
        query=self.split(self.w_q(query)) #batch*head step head_size
        key=self.split(self.w_k(key))
        value=self.split(self.w_v(value))
        d=query.shape[2]
        score=torch.bmm(query,key.transpose(1,2))/math.sqrt(d)
        if flag:
            assert query.shape[1] == key.shape[1]
            causal_mask = torch.triu(
                torch.ones(query.shape[1], key.shape[1],
                           dtype=torch.bool, device=query.device),
                diagonal=1
            )
            score = score.masked_fill(causal_mask, float("-inf"))
        valid_len=valid_len.repeat_interleave(self.head_num)
        score=mask_softmax(score,valid_len)
        score=torch.bmm(score,value)
        score=self.merge(score)
        return self.w_o(score)

        

class post_embedding(nn.Module):
    def __init__(self,max_len,embedding_size,dropout_num):
        super().__init__()
        self.embedding=nn.Embedding(max_len,embedding_size)
        self.dropout=nn.Dropout(dropout_num)
    def forward(self,x):
        size=x.shape[1]
        t=torch.arange(size,device=x.device)
        t=self.embedding(t)
        return self.dropout(x+t)
class preencoder(nn.Module):
    def __init__(self,input_num,embedding_size,pad_idx):
        super().__init__()
        self.input_num=input_num
        self.embedding_size=embedding_size
        self.embedding=nn.Embedding(input_num,embedding_size,padding_idx=pad_idx)
        self.position_embedding=post_embedding(20,embedding_size,0.1)
    def forward(self,x):
        x=self.embedding(x) #x batch step embedding_size
        x=x*math.sqrt(self.embedding_size)
        return self.position_embedding(x) #batch step embedding_size

class encoder_block(nn.Module):
    def __init__(self,query_size,key_size,value_size,head_num,attention_hidden):
        super().__init__()
        self.attention_module=attention_module(query_size,key_size,value_size,head_num,attention_hidden)
        self.dropout1=nn.Dropout(0.1)
        self.layernorm1=nn.LayerNorm(attention_hidden)
        self.fnn=nn.Sequential(nn.Linear(attention_hidden,attention_hidden*2),nn.ReLU(),nn.Linear(attention_hidden*2,attention_hidden))
        self.dropout2=nn.Dropout(0.1)
        self.layernorm2=nn.LayerNorm(attention_hidden)
    def forward(self,x,valid_len,flag):
        t=self.attention_module(x,x,x,valid_len,flag)
        y=x+self.dropout1(t)
        y=self.layernorm1(y)
        f=self.fnn(y)
        output=y+self.dropout2(f)
        output=self.layernorm2(output)
        return output #batch step attention_hidden
class transformer_encoder(nn.Module):
    def __init__(self,input_num,embedding_size,encoder_num,pad_idx):
        super().__init__()
        self.preencoder=preencoder(input_num,embedding_size,pad_idx)
        layer=[]
        for i in range(encoder_num):
            layer.append(encoder_block(embedding_size,embedding_size,embedding_size,4,embedding_size))
        self.net=nn.Sequential(*(layer))
    def forward(self,x,valid_len):
        x=self.preencoder(x)
        for layer in self.net:
            x=layer(x,valid_len,False)
        return x




编码器数据流向
输入词元编号 x 的形状为 (batch, step)。经过词嵌入后变成 (batch, step, embedding_size)，再进行缩放并加入位置编码，形状保持不变。之后依次进入多个编码器块。每个编码器块先进行多头自注意力，然后进行残差连接和层归一化；接着通过逐位置 MLP，再进行一次残差连接和层归一化。每个编码器块的输入输出形状都是 (batch, step, embedding_size)。

In [30]:
class predecoder(nn.Module):
    def __init__(self,input_num,embedding_size,pad_idx):
        super().__init__()
        self.embedding=nn.Embedding(input_num,embedding_size,padding_idx=pad_idx)
        self.post_embedding=post_embedding(20,embedding_size,0.1)
        self.embedding_size=embedding_size
    def forward(self,x):
        x=self.embedding(x)
        x=x*math.sqrt(self.embedding_size)
        x=self.post_embedding(x)
        return x #batch tar_step,embedding
class decoder_module(nn.Module):
    def __init__(self,tar_embedding_size,head_num,src_embedding):
        super().__init__()
        self.attention_module1=attention_module(tar_embedding_size,tar_embedding_size,tar_embedding_size,head_num,tar_embedding_size)
        self.dropout1=nn.Dropout(0.1)
        self.layernorm1=nn.LayerNorm(tar_embedding_size)
        self.attention_module2=attention_module(tar_embedding_size,src_embedding,src_embedding,head_num,tar_embedding_size)
        self.dropout2=nn.Dropout(0.1)
        self.layernorm2=nn.LayerNorm(tar_embedding_size)
        self.fnn=nn.Sequential(nn.Linear(tar_embedding_size,tar_embedding_size*2),nn.ReLU(),nn.Linear(tar_embedding_size*2,tar_embedding_size))
        self.dropout3=nn.Dropout(0.1)
        self.layernorm3=nn.LayerNorm(tar_embedding_size)
    def forward(self,y,y_len,encoder_output,x_len):
        sa=self.attention_module1(y,y,y,y_len,True)
        sa=self.layernorm1(y+self.dropout1(sa))
        z=self.attention_module2(sa,encoder_output,encoder_output,x_len,False)
        z=self.layernorm2(sa+self.dropout2(z))
        ffn1=self.fnn(z)
        ffn1=self.layernorm3(z+self.dropout3(ffn1))
        return ffn1
class transformer_decoder(nn.Module):
    def __init__(self,input_num,embedding_size,decoder_num,encoder_output_size,pad_idx):
        super().__init__()
        self.predecoder=predecoder(input_num,embedding_size,pad_idx)
        layer=[]
        for i in range(decoder_num):
            layer.append(decoder_module(embedding_size,4,encoder_output_size))
        self.net=nn.Sequential(*(layer))
        self.linear=nn.Linear(embedding_size,input_num)
        self.embedding_size=embedding_size
    def forward(self,encoder_output,x_len,y,y_len):
        y=self.predecoder(y)
        for layer in self.net:
            y=layer(y,y_len,encoder_output,x_len)
        y=y.reshape(-1,self.embedding_size) 
        output=self.linear(y)
        return output
class seq2seq(nn.Module):
    def __init__(self,encoder_input_num,encoder_embedding,encoder_num,decoder_input_num,decoder_embedding,decoder_num,bos_idx,src_idx,tgt_idx):
        super().__init__()
        self.transformer_encoder=transformer_encoder(encoder_input_num,encoder_embedding,encoder_num,src_idx)
        self.transformer_decoder=transformer_decoder(decoder_input_num,decoder_embedding,decoder_num,encoder_embedding,tgt_idx)
        self.bos_idx=bos_idx
    def forward(self,x,x_len,y,y_len):
        encoder_output=self.transformer_encoder(x,x_len)
        bos=torch.full(size=(y.shape[0],1),fill_value=self.bos_idx,dtype=torch.long,device=y.device)
        y=torch.cat((bos,y[:,:-1]),dim=1)
        output=self.transformer_decoder(encoder_output,x_len,y,y_len)
        return output





解码器数据流向：
上面的编码器只输出一个encoder_output。解码器输入y，这里的y是移位处理过的。y的形状（batch，step）。接着y经过pre-decoder。先进行embedding，然后加位置编码。y就变成了（batch，step，embedding）接下来进入一个decoder模块，y进行因果掩码自注意力，得到sa（batch，step，embedding）接着和y进行残差合并。得到sa （batch，step，embedding）。接下来sa与encoder_output进行交叉注意力，得到z。z再与sa进行残差合并 得到新的z。接下来把z送入mlp得到ffn，然后与z再残差合并为ffn，这个ffn的形状为（batch，step，embedding）然后作为下一个decoder模块的输入
最后经过完decoder后，reshape一下进行线性映射回字典大小。

In [31]:
net=seq2seq(len(src_vocab["token_to_idx"]),256,2,len(tgt_vocab['token_to_idx']),512,2,tgt_bos_idx,src_pad_idx,tgt_pad_idx).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
loss = nn.CrossEntropyLoss(
    ignore_index=tgt_pad_idx
)


In [32]:
for i in range(epoch):
    net.train()
    train_loss=0.0
    train_count=0
    for x,x_len,y,y_len in train_iter:
        x=x.to(device)
        y=y.to(device)
        x_len=x_len.to(device)
        y_len=y_len.to(device)
        trainer.zero_grad()
        logit=net(x,x_len,y,y_len)
        y=y.reshape(-1)
        l=loss(logit,y)
        l.backward()
        nn.utils.clip_grad_norm_(
            net.parameters(),
            max_norm=1.0,
        )
        trainer.step()
        train_count+=y_len.sum().item()
        train_loss+=l.item()*y_len.sum().item()
    print(f"epoch{i+1},loss:{train_loss/train_count:.5f}")


epoch1,loss:3.88001
epoch2,loss:2.75448
epoch3,loss:2.50682
epoch4,loss:2.38389
epoch5,loss:2.29120
epoch6,loss:2.21622
epoch7,loss:2.12581
epoch8,loss:2.05204
epoch9,loss:1.97979
epoch10,loss:1.91943
epoch11,loss:1.87413
epoch12,loss:1.82389
epoch13,loss:1.77024
epoch14,loss:1.74187
epoch15,loss:1.68971
epoch16,loss:1.64797
epoch17,loss:1.61265
epoch18,loss:1.56163
epoch19,loss:1.54047
epoch20,loss:1.48971
epoch21,loss:1.46202
epoch22,loss:1.41565
epoch23,loss:1.37519
epoch24,loss:1.36715
epoch25,loss:1.33074
epoch26,loss:1.31356
epoch27,loss:1.27658
epoch28,loss:1.25772
epoch29,loss:1.21392
epoch30,loss:1.21254
epoch31,loss:1.16549
epoch32,loss:1.16612
epoch33,loss:1.13215
epoch34,loss:1.11216
epoch35,loss:1.09448
epoch36,loss:1.09269
epoch37,loss:1.06029
epoch38,loss:1.04176
epoch39,loss:1.02429
epoch40,loss:1.00114
epoch41,loss:0.96491
epoch42,loss:0.94747
epoch43,loss:0.92542
epoch44,loss:0.91183
epoch45,loss:0.90467
epoch46,loss:0.86670
epoch47,loss:0.87916
epoch48,loss:0.86229
e

In [33]:
def predict_transformer(sentence, net, src_vocab, tgt_vocab, num_steps, device):
    net.eval()

    src_token_to_idx = src_vocab["token_to_idx"]
    tgt_token_to_idx = tgt_vocab["token_to_idx"]
    tgt_idx_to_token = tgt_vocab["idx_to_token"]

    src_unk_idx = src_token_to_idx["<unk>"]
    src_pad_idx = src_token_to_idx["<pad>"]
    src_eos_idx = src_token_to_idx["<eos>"]
    tgt_pad_idx = tgt_token_to_idx["<pad>"]
    tgt_bos_idx = tgt_token_to_idx["<bos>"]
    tgt_eos_idx = tgt_token_to_idx["<eos>"]

    sentence = preprocess_nmt(sentence)
    source_tokens = sentence.split()
    source_ids = [
        src_token_to_idx.get(token, src_unk_idx)
        for token in source_tokens
    ]
    source_ids.append(src_eos_idx)

    source_valid_len = min(len(source_ids), num_steps)
    source_ids = truncate_pad(
        source_ids,
        num_steps,
        src_pad_idx,
    )

    x = torch.tensor(
        source_ids,
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)
    x_len = torch.tensor(
        [source_valid_len],
        dtype=torch.long,
        device=device,
    )

    decoder_input = torch.tensor(
        [[tgt_bos_idx]],
        dtype=torch.long,
        device=device,
    )
    result = []

    with torch.no_grad():
        encoder_output = net.transformer_encoder(x, x_len)

        for _ in range(num_steps):
            y_len = torch.tensor(
                [decoder_input.shape[1]],
                dtype=torch.long,
                device=device,
            )

            logits = net.transformer_decoder(
                encoder_output,
                x_len,
                decoder_input,
                y_len,
            )

            last_logits = logits[-1]
            last_logits[tgt_pad_idx] = float("-inf")
            last_logits[tgt_bos_idx] = float("-inf")
            predicted_idx = last_logits.argmax().item()

            if predicted_idx == tgt_eos_idx:
                break

            result.append(tgt_idx_to_token[predicted_idx])

            next_token = torch.tensor(
                [[predicted_idx]],
                dtype=torch.long,
                device=device,
            )
            decoder_input = torch.cat(
                (decoder_input, next_token),
                dim=1,
            )

    return " ".join(result)


translation = predict_transformer(
    "go .",
    net,
    src_vocab,
    tgt_vocab,
    num_steps,
    device,
)
print(translation)


va !
